# Seminar 11. Local LLM Inference

The following examples demonstrate how to infer models locally via [LM Studio](https://lmstudio.ai/) server.

## Simple Usage

For starters, let's ask the model to write some code in python

In [ ]:
#pip install openai


In [1]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

def ask_local_model(prompt):
    completion = client.chat.completions.create(
      model="local-model", # LM Studio uses the currently loaded model
      messages=[
        {"role": "system", "content": "You are a Python programmer."},
        {"role": "user", "content": prompt}
      ],
      temperature=0.7,
    )
    return completion.choices[0].message.content

print(ask_local_model("Please, write a simple function for counting Fibonacci srquence"))


<think>
Okay, the user wants me to write a simple function to count the Fibonacci sequence. Let me think about how to approach this.

First, I need to recall what the Fibonacci sequence is. It starts with 0 and 1, and each subsequent number is the sum of the two preceding ones. So the sequence goes 0, 1, 1, 2, 3, 5, 8, etc.

Now, the user said "counting Fibonacci sequence". Maybe they want a function that generates the sequence up to a certain number or returns the nth term. But the wording is a bit unclear. Let me check the query again: "write a simple function for counting Fibonacci srquence". Hmm, maybe they mean generate the sequence up to a given number of terms.

So I should create a function that takes an integer n and returns the first n numbers in the Fibonacci sequence. Alternatively, maybe it's about calculating the nth Fibonacci number. But since the user said "counting", perhaps generating the sequence is more appropriate.

Let me think about possible approaches. The simpl

In [ ]:
def fibonacci(n):
    """
    Generate the first n numbers in the Fibonacci sequence.
    
    The Fibonacci sequence starts with 0 and 1, and each subsequent number is the sum of the two preceding ones.
    
    Parameters:
    n (int): Number of terms to generate. If n <= 0, returns an empty list.
    
    Returns:
    list: A list containing the first n Fibonacci numbers.
    """
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    fib_sequence = [0, 1]
    for i in range(2, n):
        next_num = fib_sequence[-1] + fib_sequence[-2]
        fib_sequence.append(next_num)
    return fib_sequence

print(fibonacci(5))  # Output: [0, 1, 1, 2, 3]
print(fibonacci(10))  # Output: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


[0, 1, 1, 2, 3]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


Cool, it works! However, we had to wait for 5,5 minutes for the answer.

## Streaming Response

In [3]:
def stream_local_model(prompt):
    print("Generating response: ", end="", flush=True)
    
    stream = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user", "content": prompt}],
        stream=True, # turn on streaming
    )
    
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            print(chunk.choices[0].delta.content, end="", flush=True)

stream_local_model("Tell me a funny joke.")


Generating response: <think>
Okay, the user asked for a funny joke. Let me think of some common ones that are light-hearted and not too complicated.

Maybe start with a classic one. There's the one about the man who goes to the doctor and says he's been feeling unwell. The doctor says, "You've been feeling unwell?" and he replies, "Yes, I've been feeling unwell." That's a bit of a play on words, but maybe too simple.

Alternatively, there's the joke about the chicken crossing the road. Wait, that's usually the one where the chicken crosses the road to get to the other side, but the punchline is that it's a joke about the chicken crossing the road to get to the other side. But maybe that's too well-known and not funny enough.

What about a pun on "I'll be there in 5 minutes"? The person says they'll be there in five minutes, then the other person says, "That's a long time," and the first person replies, "No, just five minutes." That's a classic one, but maybe it's too straightforward.



KeyboardInterrupt: 

## Creating Local Agent with `smolagents`

In [ ]:
#pip install "smolagents[openai]" ddgs ipywidgets


In [1]:
from smolagents import CodeAgent, OpenAIModel
from tqdm.auto import tqdm

local_model = OpenAIModel(
    model_id="local-model", # name doesn't matter
    api_base="http://localhost:1234/v1", # local LM Studio server
    api_key="lm-studio"
)

agent = CodeAgent(
    tools=[],
    model=local_model,
    add_base_tools=True
)

agent.run("Сколько будет 256 умножить на 1024 и вычти из этого 500?")


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Сколько будет 256 умножить на 1024 и вычти из этого 500?                                                        │
│                                                                                                                 │
╰─ OpenAIModel - local-model ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = (256 * 1024) - 500                                                                                      
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 261644

[Step 1: Duration 108.23 seconds| Input tokens: 2,158 | Output tokens: 367]

261644